# 01 Data pipeline: from raw prices to a stocks x days return matrix

**Project:** Do Sectors Describe How Stocks Move?

This notebook builds the dataset used by every later analysis:

1. Scrape the S&P 500 list with GICS sectors from Wikipedia (BeautifulSoup)
2. Download about two years of daily prices from Yahoo Finance (`yfinance`)
3. Clean and transform with one **lazy Polars** query: join, drop stocks with missing history, compute daily returns
4. Pivot to a **stocks x days** matrix of returns
5. Sanity checks

The functions live in `src/data.py`; this notebook calls them and explains each step.
Downloads are saved in `data/raw/` as Parquet files, so rerunning the notebook does not download again.

## Setup

**Why Polars instead of pandas?** Three differences matter here:

- **No index.** A pandas DataFrame has a row index (dates, tickers, ...). A Polars DataFrame is just columns. Grouping and ordering are always written out (`group_by`, `sort`, `.over`).
- **Expressions.** Polars code is built from expressions such as `pl.col("close").pct_change()`. An expression is a recipe for a column, not a result.
- **Contexts.** An expression runs inside a context, which decides what happens to the result:
  - `select(...)`: keep only the columns you list
  - `with_columns(...)`: add or replace columns, keep the rest
  - `filter(...)`: keep the rows where the condition is true
  - `group_by(...).agg(...)`: one row per group

In [1]:
import sys                       # to tell Python where to find our own code
from datetime import timedelta   # to look a few days before/after a date

import polars as pl              # data processing

sys.path.append("..")            # the project folder (one level up) holds the src/ package
from src import config, data     # our settings and pipeline functions

pl.Config.set_tbl_rows(12)       # show at most 12 rows when printing a table

print("Date range:", config.START_DATE, "to", config.END_DATE)
print("Missing-data threshold:", config.MISSING_THRESHOLD)

Date range: 2024-11-18 to 2026-11-18
Missing-data threshold: 0.05


## 1. Sector list (Wikipedia, BeautifulSoup)

`data.fetch_sectors()` does four things:

1. Downloads the Wikipedia page with `requests`. Wikipedia blocks the default Python User-Agent, so we send our own.
2. Parses the HTML with **BeautifulSoup** and finds the table with `id="constituents"`.
3. Loops over the table rows (`<tr>`) and reads the first four cells (`<td>`): ticker, company, sector, sub-industry.
4. Builds a Polars DataFrame from that list of rows.

Tickers are converted to Yahoo's format: Wikipedia writes share classes with a dot (`BRK.B`), Yahoo uses a dash (`BRK-B`).

Note: this is today's index membership. Companies that left the index during the two years are not included (survivorship bias, accepted for this project).

In [2]:
sectors = data.fetch_sectors()   # scrape Wikipedia, or load the saved copy
print(sectors.shape)             # (rows, columns)
sectors.head()                   # first 5 rows

(503, 4)


ticker,company,sector,sub_industry
str,str,str,str
"""A""","""Agilent Technologies""","""Health Care""","""Life Sciences Tools & Services"""
"""AAPL""","""Apple Inc.""","""Information Technology""","""Technology Hardware, Storage &…"
"""ABBV""","""AbbVie""","""Health Care""","""Biotechnology"""
"""ABNB""","""Airbnb""","""Consumer Discretionary""","""Hotels, Resorts & Cruise Lines"""
"""ABT""","""Abbott Laboratories""","""Health Care""","""Health Care Equipment"""


In [3]:
# Tickers that contain a dash: the share classes that were converted
sectors.filter(pl.col("ticker").str.contains("-"))

ticker,company,sector,sub_industry
str,str,str,str
"""BF-B""","""Brown–Forman""","""Consumer Staples""","""Distillers & Vintners"""
"""BRK-B""","""Berkshire Hathaway""","""Financials""","""Multi-Sector Holdings"""


Counting stocks per sector uses the **`group_by` / `agg` context**. In pandas, `df.groupby("sector").size()` returns a Series with `sector` as the index. In Polars the result is a normal DataFrame with `sector` as an ordinary column.

In [4]:
sector_counts = sectors.group_by("sector").agg(pl.len().alias("n_stocks"))  # rows per sector
sector_counts.sort("n_stocks", descending=True)                             # biggest first

sector,n_stocks
str,u32
"""Industrials""",83
"""Financials""",76
"""Information Technology""",76
"""Health Care""",60
"""Consumer Discretionary""",47
"""Consumer Staples""",33
"""Utilities""",31
"""Real Estate""",30
"""Materials""",24


## 2. Prices (yfinance)

`data.fetch_prices()` downloads daily prices in batches of 100 tickers with `auto_adjust=True`, so prices are adjusted for splits and dividends (otherwise a 2-for-1 split would look like a -50% day).

yfinance returns a wide pandas table with one column per ticker. We convert it to Polars and **unpivot** it to long format: one row per `(date, ticker)`. A day without a price becomes a row with `close = null`, so missing data stays visible.

The first run downloads (about a minute). Later runs load `data/raw/prices.parquet`.

In [5]:
tickers = sectors["ticker"].to_list()   # list of all ticker symbols
prices = data.fetch_prices(tickers)     # download, or load the saved copy
print(prices.shape)
print("Trading days:", prices["date"].n_unique())
print("From", prices["date"].min(), "to", prices["date"].max())
prices.head()

(252506, 4)
Trading days: 502
From 2024-11-18 to 2026-11-18


date,ticker,close,volume
date,str,f64,f64
2024-11-18,"""A""",124.961037,1.445e6
2024-11-19,"""A""",123.750023,1.8382e6
2024-11-20,"""A""",126.585564,1.696e6
2024-11-21,"""A""",130.021698,2.8821e6
2024-11-22,"""A""",131.774231,2.0678e6


## 3. Transform with the lazy API

`data.build_returns_pipeline()` describes the whole transformation as **one lazy query**:

1. Remove duplicate share classes. Alphabet, Fox and News Corp are each listed twice (e.g. GOOG and GOOGL). Both tickers are the same company and move almost identically, so we keep the Class A ticker (GOOGL, FOXA, NWSA) and drop the other (list in `config.DUPLICATE_SHARE_CLASSES`)
2. Join prices with sectors on `ticker`
3. For each stock, count the days with a price and compute the share of missing days; drop stocks above 5%
4. Sort by ticker and date, compute daily returns with `pl.col("close").pct_change().over("ticker")`
5. Drop the first day of each stock (no previous price, so no return)

**Lazy vs eager.** With a normal (eager) DataFrame every line runs immediately. With a `LazyFrame`, Polars only writes the steps down. Nothing runs until `.collect()`. Because Polars sees the whole query first, it can rewrite it to do less work, without changing the result.

We start from `pl.scan_parquet` (a lazy read of the saved files), so the plan also shows what is read from disk.

In [6]:
prices_lazy = pl.scan_parquet(config.RAW_PRICES_PATH)     # lazy: file is not read yet
sectors_lazy = pl.scan_parquet(config.RAW_SECTORS_PATH)   # lazy: file is not read yet
lazy_returns = data.build_returns_pipeline(prices_lazy, sectors_lazy)  # only builds the plan
type(lazy_returns)                                         # a LazyFrame, not data

polars.lazyframe.frame.LazyFrame

### The plan as written: `.explain(optimized=False)`

Read plans **from the bottom up**: the bottom is where data comes in, the top is the final result.

In [7]:
print(lazy_returns.explain(optimized=False))   # the steps exactly as we wrote them

SELECT [col("date"), col("ticker"), col("company"), col("sector"), col("close"), col("ret")]
  FILTER col("ret").is_not_null()
  FROM
     WITH_COLUMNS:
     [col("close").pct_change([dyn int: 1]).over([col("ticker")]).alias("ret")] 
      SORT BY [col("ticker"), col("date")]
        FILTER col("missing_share") <= 0.05
        FROM
           WITH_COLUMNS:
           [(1.0 - (col("n_prices") / col("n_days"))).alias("missing_share")] 
             WITH_COLUMNS:
             [col("close").count().over([col("ticker")]).alias("n_prices"), col("date").n_unique().alias("n_days")] 
              INNER JOIN:
              LEFT PLAN ON: [col("ticker")]
                FILTER col("ticker").is_in([["GOOG", "FOX", "NWS"]]).not()
                FROM
                  Parquet SCAN [/Users/geraldchan/Desktop/SnP500-Sector-Analysis/data/raw/prices.parquet]
                  PROJECT */4 COLUMNS
                  ESTIMATED ROWS: 252506
              RIGHT PLAN ON: [col("ticker")]
                Parque

### The plan Polars will run: `.explain()`

In [8]:
print(lazy_returns.explain())   # the steps after Polars optimised them

FILTER col("ret").is_not_null()
FROM
  simple π 6/6 ["date", "ticker", "company", ... 3 other columns]
     WITH_COLUMNS:
     [col("close").pct_change([dyn int: 1]).over([col("ticker")]).alias("ret")] 
      SORT BY [col("ticker"), col("date")]
        simple π 5/5 ["date", "ticker", "close", ... 2 other columns]
          FILTER col("missing_share") <= 0.05
          FROM
            simple π 6/6 ["date", "ticker", "close", ... 3 other columns]
               WITH_COLUMNS:
               [(1.0 - (col("n_prices") / col("n_days"))).alias("missing_share")] 
                 WITH_COLUMNS:
                 [col("close").count().over([col("ticker")]).alias("n_prices"), col("date").n_unique().alias("n_days")] 
                  INNER JOIN:
                  BUILD SIDE: PreferRight
                  LEFT PLAN ON: [col("ticker")]
                    Parquet SCAN [/Users/geraldchan/Desktop/SnP500-Sector-Analysis/data/raw/prices.parquet]
                    PROJECT 3/4 COLUMNS
                 

**What changed:**

- **Projection pushdown** (reading fewer columns). The plan as written reads all 4 columns of each file (`PROJECT */4 COLUMNS`). The optimised plan reads only 3 (`PROJECT 3/4 COLUMNS`): `volume` and `sub_industry` are never used, so they are never read from disk.
- **Predicate pushdown** (filtering early). The filter that removes duplicate share classes moved into the file scan of prices (`SELECTION: ...`), so those rows are skipped while the file is read instead of being loaded and removed afterwards.
- **Helper columns dropped early.** New steps marked `simple π` remove columns as soon as they are no longer needed: `n_prices` and `n_days` right after `missing_share` is computed, then `missing_share` right after the filter. Less data has to be sorted.
- **Join strategy.** `BUILD SIDE: PreferRight` means Polars builds its lookup table from the small sectors table (503 rows) and streams the big prices table (about 250k rows) through it.
- **What did *not* move.** The `missing_share <= 0.05` filter stays where it is. It cannot run earlier because a stock's missing share needs all of its rows first. Polars only makes changes that cannot change the result.

A smaller example of predicate pushdown on its own: keep one ticker. The filter again moves into the file scan (`SELECTION: ...`).

In [9]:
one_stock = (
    pl.scan_parquet(config.RAW_PRICES_PATH)    # lazy read
    .select("date", "ticker", "close")         # three columns
    .filter(pl.col("ticker") == "AAPL")        # only Apple
)
print(one_stock.explain())                     # look for SELECTION in the scan

Parquet SCAN [/Users/geraldchan/Desktop/SnP500-Sector-Analysis/data/raw/prices.parquet]
PROJECT 3/4 COLUMNS
SELECTION: col("ticker") == "AAPL"
ESTIMATED ROWS: 252506


Now run the query. `.collect()` executes the optimised plan and returns a normal DataFrame.

In [10]:
returns_long = lazy_returns.collect()   # run the plan
print(returns_long.shape)
returns_long.head()

(247995, 6)


date,ticker,company,sector,close,ret
date,str,str,str,f64,f64
2024-11-19,"""A""","""Agilent Technologies""","""Health Care""",123.750023,-0.009691
2024-11-20,"""A""","""Agilent Technologies""","""Health Care""",126.585564,0.022913
2024-11-21,"""A""","""Agilent Technologies""","""Health Care""",130.021698,0.027145
2024-11-22,"""A""","""Agilent Technologies""","""Health Care""",131.774231,0.013479
2024-11-25,"""A""","""Agilent Technologies""","""Health Care""",132.414215,0.004857


### Why `.over("ticker")` matters

`pct_change()` compares each row with the row above it. Our table has all stocks stacked under each other, sorted by ticker and date. Without `.over("ticker")`, the first day of each stock is compared with the **last day of the stock above it**, which gives a nonsense return.

`.over("ticker")` is a **window function**: it runs the calculation separately inside each ticker's rows. pandas would need `df.groupby("ticker")["close"].pct_change()`; in Polars it is just part of the expression.

Below: the same calculation with and without `.over`, where stock A ends and AAPL starts.

In [11]:
two_stocks = prices.filter(pl.col("ticker").is_in(["A", "AAPL"]))   # just two stocks
two_stocks = two_stocks.sort("ticker", "date")                      # A first, then AAPL

two_stocks = two_stocks.with_columns(
    pl.col("close").pct_change().alias("ret_wrong"),                 # ignores where a stock ends
    pl.col("close").pct_change().over("ticker").alias("ret_right"),  # restarts for each stock
)

n_rows_a = two_stocks.filter(pl.col("ticker") == "A").height   # how many rows stock A has
two_stocks.slice(n_rows_a - 2, 4)   # last 2 rows of A and first 2 rows of AAPL

date,ticker,close,volume,ret_wrong,ret_right
date,str,f64,f64,f64,f64
2026-11-17,"""A""",172.220001,2.4653e6,-0.004048,-0.004048
2026-11-18,"""A""",177.330002,2.4889e6,0.029671,0.029671
2024-11-18,"""AAPL""",226.205292,4.46337e7,0.275618,null
2024-11-19,"""AAPL""",226.463226,3.62118e7,0.00114,0.00114


In `ret_wrong`, AAPL's first day gets a "return" computed from A's last price: a made-up move. In `ret_right` that row is correctly null (there is no earlier AAPL price), and the pipeline then drops it. Across 500 stocks the wrong version would add about 500 fake returns.

## 4. Pivot to a stocks x days matrix

scikit-learn expects a table where **each row is one sample** and **each column is one feature**. We want to group stocks, so:

- rows = stocks
- columns = trading days: each stock is described by its daily returns

Two stocks that move together have similar rows, which is what correlation, PCA and KMeans will pick up later.

`pivot(on="date", index="ticker", values="ret")` turns the long table (one row per stock per day) into this wide table. Polars has no index, so `ticker` stays a normal column; we split it off before giving the numbers to scikit-learn.

In [12]:
matrix_raw = data.pivot_returns(returns_long)   # long -> wide
print(matrix_raw.shape)                         # (stocks, 1 ticker column + one column per day)
matrix_raw.select(matrix_raw.columns[:6]).head()   # ticker and the first 5 days

(495, 502)


ticker,2024-11-19,2024-11-20,2024-11-21,2024-11-22,2024-11-25
str,f64,f64,f64,f64,f64
"""A""",-0.009691,0.022913,0.027145,0.013479,0.004857
"""AAPL""",0.00114,0.003154,-0.002096,0.005907,0.013051
"""ABBV""",0.001744,0.007144,0.023665,0.030397,0.000621
"""ABNB""",-0.005674,0.028987,-0.014714,0.030617,0.026139
"""ABT""",-0.00196,-0.010245,0.011473,0.004264,0.003142


### Nulls after the pivot

If a stock has no return on some day, that cell of the matrix is empty (null). How many are there?

In [13]:
nulls = data.null_report(matrix_raw)   # empty cells per ticker
print("Stocks with empty cells:", nulls.height)
nulls

Stocks with empty cells: 0


ticker,n_null_days
str,u32


**Options for empty cells, and the trade-offs:**

| Option | Effect | Trade-off |
|---|---|---|
| Fill with 0 | Treat a missing day as "no move" | Keeps every stock and day. Slightly weakens that stock's correlations, which only matters if many cells are filled |
| Drop dates with any empty cell | Keep only days every stock traded | No made-up values, but one stock's gap removes that day for all 500 stocks |
| Drop stocks with any empty cell | Keep only complete stocks | No made-up values, but loses stocks |
| Fill with the sector's return | A plausible value | Makes the stock look like its sector, which biases exactly what we want to test |

**Choice: fill with 0** (`data.fill_nulls_with_zero`), the simplest defensible option.

**In this dataset it changes nothing:** the 5% rule already dropped every stock with missing history, and the kept stocks have a price on every trading day. The fill stays in the pipeline as a safeguard in case the dates or the threshold change.

In [14]:
matrix = data.fill_nulls_with_zero(matrix_raw)        # replace empty cells with 0
print("Empty cells left:", data.null_report(matrix).height)
print("Matrix shape:", matrix.shape)

Empty cells left: 0
Matrix shape: (495, 502)


## 5. Sanity checks
### Stocks kept and dropped

In [15]:
report = data.missing_report(prices, sectors)        # one row per ticker
dropped = report.filter(pl.col("kept") == False)     # the stocks that failed the 5% rule

print("Kept:", report.height - dropped.height, "| Dropped:", dropped.height)
dropped.select("ticker", "company", "sector", "n_prices", "missing_share", "first_price", "reason")

Kept: 498 | Dropped: 5


ticker,company,sector,n_prices,missing_share,first_price,reason
str,str,str,u32,f64,date,str
"""VYLR""","""Vylor""","""Consumer Staples""",35,0.930279,2026-10-01,"""prices start late (new listing…"
"""HONA""","""Honeywell Aerospace""","""Industrials""",110,0.780876,2026-06-15,"""prices start late (new listing…"
"""FDXF""","""FedEx Freight""","""Industrials""",123,0.75498,2026-05-27,"""prices start late (new listing…"
"""Q""","""Qnity Electronics""","""Information Technology""",268,0.466135,2025-10-27,"""prices start late (new listing…"
"""SNDK""","""Sandisk""","""Information Technology""",444,0.115538,2025-02-13,"""prices start late (new listing…"


On top of these, the duplicate share classes were removed, which gives the final number of stocks:

In [16]:
print("Duplicate share classes removed:", config.DUPLICATE_SHARE_CLASSES)
print("Stocks in the final matrix:", matrix.height)

Duplicate share classes removed: ['GOOG', 'FOX', 'NWS']
Stocks in the final matrix: 495


### Stocks per sector after filtering

In [17]:
kept = returns_long.select("ticker", "sector").unique()   # one row per kept stock
kept.group_by("sector").agg(pl.len().alias("n_stocks")).sort("n_stocks", descending=True)

sector,n_stocks
str,u32
"""Industrials""",81
"""Financials""",76
"""Information Technology""",74
"""Health Care""",60
"""Consumer Discretionary""",47
"""Consumer Staples""",32
"""Utilities""",31
"""Real Estate""",30
"""Materials""",24


### Date range and number of trading days

There is one fewer return day than price day: the first day has no previous price, so no return.

In [18]:
print("First return day:", returns_long["date"].min())
print("Last return day: ", returns_long["date"].max())
print("Return days:     ", returns_long["date"].n_unique())
print("Stocks:          ", returns_long["ticker"].n_unique())

First return day: 2024-11-19
Last return day:  2026-11-18
Return days:      501
Stocks:           495


### Spot check: pipeline returns vs a manual calculation

For AAPL's first few days we compute `close_today / close_yesterday - 1` by hand in plain Python and compare it with the pipeline. The difference should be tiny (around 1e-16, the rounding of decimal numbers in a computer).

In [19]:
check = data.spot_check_return(prices, returns_long, "AAPL")
print("Largest difference:", check["difference"].max())
check

Largest difference: 9.280770596475918e-17


date,prev_close,close,manual_ret,ret,difference
date,f64,f64,f64,f64,f64
2024-11-19,226.205292,226.463226,0.00114,0.00114,3.7513e-17
2024-11-20,226.463226,227.177475,0.003154,0.003154,3.5128e-17
2024-11-21,227.177475,226.70134,-0.002096,-0.002096,5.4644e-17
2024-11-22,226.70134,228.040573,0.005907,0.005907,9.2808e-17
2024-11-25,228.040573,231.016708,0.013051,0.013051,7.9797e-17


### Extreme returns (more than 50% up or down in one day)

In [20]:
extremes = data.extreme_returns(returns_long)
extremes

date,ticker,company,sector,close,ret
date,str,str,str,f64,f64
2025-08-26,"""ECHO""","""EchoStar""","""Communication Services""",50.869999,0.702477
2026-08-19,"""MRNA""","""Moderna""","""Health Care""",174.380005,1.769695


For each extreme day we look at the prices around it. A data error would jump and then snap back; a real event moves the price to a new level that stays.

In [21]:
for row in extremes.iter_rows(named=True):     # one extreme day at a time
    start = row["date"] - timedelta(days=5)    # a few days before
    end = row["date"] + timedelta(days=7)      # a few days after
    around = returns_long.filter(
        (pl.col("ticker") == row["ticker"]) & pl.col("date").is_between(start, end)
    )
    print(row["ticker"], row["company"])
    print(around.select("date", "close", "ret"))

ECHO EchoStar
shape: (8, 3)
┌────────────┬───────────┬───────────┐
│ date       ┆ close     ┆ ret       │
│ ---        ┆ ---       ┆ ---       │
│ date       ┆ f64       ┆ f64       │
╞════════════╪═══════════╪═══════════╡
│ 2025-08-21 ┆ 27.5      ┆ -0.005425 │
│ 2025-08-22 ┆ 29.620001 ┆ 0.077091  │
│ 2025-08-25 ┆ 29.879999 ┆ 0.008778  │
│ 2025-08-26 ┆ 50.869999 ┆ 0.702477  │
│ 2025-08-27 ┆ 58.759998 ┆ 0.155101  │
│ 2025-08-28 ┆ 56.950001 ┆ -0.030803 │
│ 2025-08-29 ┆ 61.790001 ┆ 0.084987  │
│ 2025-09-02 ┆ 65.239998 ┆ 0.055834  │
└────────────┴───────────┴───────────┘
MRNA Moderna
shape: (9, 3)
┌────────────┬────────────┬───────────┐
│ date       ┆ close      ┆ ret       │
│ ---        ┆ ---        ┆ ---       │
│ date       ┆ f64        ┆ f64       │
╞════════════╪════════════╪═══════════╡
│ 2026-08-14 ┆ 63.32      ┆ -0.005185 │
│ 2026-08-17 ┆ 64.459999  ┆ 0.018004  │
│ 2026-08-18 ┆ 62.959999  ┆ -0.02327  │
│ 2026-08-19 ┆ 174.380005 ┆ 1.769695  │
│ 2026-08-20 ┆ 133.320007 ┆ -0.235463 │

Both jumps are followed by prices that stay at the new level, so they look like real events (news), not data errors. They are kept. Single large days can still affect a stock's correlations, which is one reason the clustering step will standardise returns.

Overall distribution of daily returns, as a final check that values look sensible:

In [22]:
returns_long["ret"].describe()   # count, mean, std, min, quartiles, max

statistic,value
str,f64
"""count""",247995.0
"""null_count""",0.0
"""mean""",0.000599
"""std""",0.023902
"""min""",-0.440438
"""25%""",-0.009972
"""50%""",0.000519
"""75%""",0.011137
"""max""",1.769695


## 6. Save outputs

- `data/processed/returns_long.parquet`: one row per stock per day (date, ticker, company, sector, close, ret)
- `data/processed/returns_matrix.parquet`: stocks x days matrix
- `data/processed/sectors.parquet`: sectors of the kept stocks

In [23]:
data.save_processed(returns_long, matrix, sectors)   # write the three files

print(pl.read_parquet(config.RETURNS_LONG_PATH).shape)     # check they were saved
print(pl.read_parquet(config.RETURNS_MATRIX_PATH).shape)
print(pl.read_parquet(config.SECTORS_PATH).shape)

(247995, 6)


(495, 502)
(495, 4)


## Summary

- 503 S&P 500 companies scraped, all downloaded from yfinance
- 5 stocks dropped for missing history (all listed or spun off during the two years)
- 3 duplicate share classes dropped (one ticker kept per company), leaving 495 stocks
- 501 trading days of returns, no empty cells in the matrix
- Returns match a manual calculation
- Two extreme daily moves checked and kept as real events